# Đề tài 2 — Hoàn thiện: Detection + Localization

Notebook này chạy phần còn thiếu theo đúng thứ tự, **không dùng test set để tuning**:

| Bước | Việc | Đầu ra |
|---|---|---|
| 0 | Môi trường, repo, đường dẫn | — |
| 1 | Hoàn tất 9 lượt U-Net–ResNet34 (RGB / Noise / DCT × seed 7, 21, 42) | `best_dice.pt` |
| 2 | Chọn ngưỡng Real/Fake trên **validation** | `thresholds.json` cạnh checkpoint |
| 3 | Đánh giá **CASIA test** | pixel + image-level metrics, confusion matrix, ảnh minh hoạ |
| 4 | Đánh giá **cross-dataset** (Coverage, zero-shot) | bảng generalization |
| 5 | Nâng cấp **SegFormer** (Transformer) cùng protocol | so sánh CNN vs Transformer |
| 6 | Tổng hợp bảng mean ± SD | `reports/tables/*.md` |
| 7 | Dự đoán 1 ảnh + **web demo** | mask, overlay, bbox, diện tích, Real/Fake |

> Chạy **cell startup cũ** của bạn trước (restore CASIA2 + manifests vào `/content`), vì manifest dùng đường dẫn tuyệt đối.

In [ ]:
# 0.1 GPU + Drive
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 0.2 Lấy code mới nhất và cài đặt (giữ nguyên torch đang có trên Colab)
REPO = '/content/image-forgery-localization'
%cd $REPO
!git pull
!pip install -q -e . --no-deps --ignore-requires-python
!pip install -q "segmentation-models-pytorch==0.6.0" gradio pytest
import torch, segmentation_models_pytorch as smp, sys
print(sys.version, torch.__version__, smp.__version__, torch.cuda.is_available())

In [ ]:
# 0.3 ĐƯỜNG DẪN — sửa cho khớp với Drive/manifest của bạn
from pathlib import Path
DRIVE = Path('/content/drive/MyDrive/ImageForgeryResearch')
MANIFEST_DIR = Path(LOCAL_MANIFEST) if 'LOCAL_MANIFEST' in globals() else Path('/content/manifests')  # chứa train/val/test.csv
ROOT = '/'                                                     # manifest dùng đường dẫn tuyệt đối
UNET_ROOT = DRIVE / 'checkpoints' / 'controlled_multiseed_v1'  # {mode}_seed{seed}/best_dice.pt
SEGF_ROOT = DRIVE / 'checkpoints' / 'topic2_segformer_v1'
EVAL_DIR = DRIVE / 'reports' / 'eval'
TABLE_DIR = DRIVE / 'reports' / 'tables'
TRAIN, VAL, TEST = (MANIFEST_DIR / f'{s}.csv' for s in ('train', 'val', 'test'))
for p in (TRAIN, VAL, TEST, UNET_ROOT):
    print(p, p.exists())

In [ ]:
# 0.4 Unit tests (synthetic, không cần dữ liệu thật)
!python -m pytest -q

## Bước 1 — Hoàn tất seed 42 cho U-Net–ResNet34

Ưu tiên: chạy lại các cell **M1 → M5** và cell seed 42 trong notebook cũ (cùng protocol, tự resume từ `last.pt`).
Cách thay thế: `scripts/train_multiinput.py` (cùng loss / augmentation / cách chọn checkpoint, nhưng thư mục tên `unet_resnet34_<mode>_seed42`).

In [ ]:
# 1.1 Kiểm tra trạng thái 9 lượt
import pandas as pd
rows = []
for seed in (7, 21, 42):
    for mode in ('rgb', 'noise', 'frequency'):
        d = UNET_ROOT / f'{mode}_seed{seed}'
        h = d / 'history.csv'
        epochs = int(pd.read_csv(h)['epoch'].max()) if h.exists() else 0
        rows.append(dict(mode=mode, seed=seed, epochs=epochs, best=(d / 'best_dice.pt').exists()))
status = pd.DataFrame(rows); display(status)

In [ ]:
# 1.2 (Tuỳ chọn) Train seed 42 bằng script nếu không dùng notebook cũ
# OUT = DRIVE / 'checkpoints' / 'unet_script_v1'
# for mode in ('rgb', 'noise', 'frequency'):
#     !python scripts/train_multiinput.py --architecture unet_resnet34 --mode $mode --seed 42 \
#         --train-manifest $TRAIN --val-manifest $VAL --root $ROOT --output-root $OUT

## Bước 2 — Chọn ngưỡng Real/Fake trên VALIDATION

Ảnh là *Forged* nếu `image_score` (trung bình 1% pixel nghi ngờ nhất) ≥ ngưỡng. Ngưỡng chọn để tối đa F1 trên val, lưu `thresholds.json` cạnh checkpoint và **đóng băng** cho test và cross-dataset.

In [ ]:
UNET_CKPTS = sorted(str(p) for p in UNET_ROOT.glob('*_seed*/best_dice.pt'))
print(len(UNET_CKPTS), 'checkpoints'); print(*UNET_CKPTS, sep='\n')
ck = ' '.join(UNET_CKPTS)
!python scripts/evaluate.py --checkpoint $ck --manifest $VAL --root $ROOT --dataset-name casia_val --calibrate --output-dir $EVAL_DIR

## Bước 3 — CASIA test (chỉ chạy sau khi đã chốt checkpoint + ngưỡng)

In [ ]:
!python scripts/evaluate.py --checkpoint $ck --manifest $TEST --root $ROOT --dataset-name casia_test \
    --isolation-manifests $TRAIN $VAL --save-visuals 8 --output-dir $EVAL_DIR

## Bước 4 — Cross-dataset: Coverage (zero-shot: không fine-tune, không đổi ngưỡng)

1. Tải Coverage → 2. tạo manifest + ảnh preview → 3. **mở preview kiểm tra** mask trắng = vùng giả và khớp ảnh → 4. chạy lại với `--masks-checked` → 5. đánh giá.
Có thể thêm Columbia / IMD2020 / NIST16 bằng preset `generic` (sau khi chuyển mask về dạng nhị phân, trắng = vùng giả).

In [ ]:
!git clone -q https://github.com/wenbihan/coverage.git /content/coverage_repo || true
!find /content/coverage_repo -maxdepth 2 | head -20

In [ ]:
COVERAGE = '/content/coverage_repo'   # sửa thành thư mục chứa image/ và mask/ (giải nén nếu là .zip)
COV_CSV = '/content/manifests_external/coverage_test.csv'
!python scripts/build_external_manifest.py coverage --dataset-root $COVERAGE --output $COV_CSV --preview /content/coverage_preview.png
from IPython.display import Image as IPImage; display(IPImage('/content/coverage_preview.png'))

In [ ]:
# Sau khi đã xem preview và xác nhận mask đúng (mask không nhị phân thì thêm --mask-threshold 127):
!python scripts/build_external_manifest.py coverage --dataset-root $COVERAGE --output $COV_CSV --masks-checked
!python scripts/evaluate.py --checkpoint $ck --manifest $COV_CSV --root $ROOT --dataset-name coverage --save-visuals 6 --output-dir $EVAL_DIR

## Bước 5 — Nâng cấp Transformer: SegFormer (cùng protocol)

Mặc định: `segformer_mit_b2`, input RGB, 3 seed. Nếu còn thời gian/GPU, thêm input tốt nhất trong Noise/DCT (chọn theo **validation**, không theo test).
Mỗi lượt tự resume từ `last.pt` nếu Colab bị ngắt.

In [ ]:
SEGF_MODES = ['rgb']          # thêm 'noise' hoặc 'frequency' nếu muốn
for seed in (7, 21, 42):
    for mode in SEGF_MODES:
        !python scripts/train_multiinput.py --architecture segformer_mit_b2 --mode $mode --seed $seed \
            --train-manifest $TRAIN --val-manifest $VAL --root $ROOT --output-root $SEGF_ROOT

In [ ]:
SEGF_CKPTS = ' '.join(sorted(str(p) for p in SEGF_ROOT.glob('*_seed*/best_dice.pt')))
!python scripts/evaluate.py --checkpoint $SEGF_CKPTS --manifest $VAL --root $ROOT --dataset-name casia_val --calibrate --output-dir $EVAL_DIR
!python scripts/evaluate.py --checkpoint $SEGF_CKPTS --manifest $TEST --root $ROOT --dataset-name casia_test --isolation-manifests $TRAIN $VAL --save-visuals 8 --output-dir $EVAL_DIR
!python scripts/evaluate.py --checkpoint $SEGF_CKPTS --manifest $COV_CSV --root $ROOT --dataset-name coverage --save-visuals 6 --output-dir $EVAL_DIR

## Bước 6 — Bảng kết quả cho báo cáo

In [ ]:
!python scripts/summarize_results.py --eval-dir $EVAL_DIR --output-dir $TABLE_DIR
pd.read_csv(TABLE_DIR / 'paired_vs_rgb.csv').round(4)

## Bước 7 — Dự đoán 1 ảnh và web demo

Chọn checkpoint tốt nhất **theo validation** (không theo test). Demo cho phép chọn giữa nhiều model trên cùng một ảnh.

In [ ]:
BEST = UNET_ROOT / 'rgb_seed7' / 'best_dice.pt'     # sửa theo kết quả validation
!python scripts/predict.py --checkpoint $BEST --image /content/sample.jpg --output-dir /content/predictions

In [ ]:
DEMO = ' '.join(str(UNET_ROOT / f'{m}_seed7' / 'best_dice.pt') for m in ('rgb', 'noise', 'frequency'))
!python app/demo.py --checkpoint $DEMO --share